# NSGA-II vs NSGA-II Partitioned (6 niches) — Kaggle Notebook

This notebook benchmarks **NSGA2** vs **NSGA2-PARTITIONED** from the repo. It:

1. Clones the repository
2. Fetches HW-NAS-Bench hardware metrics (`data/HW-NAS-Bench-v1_0.pickle`)
3. (Optional) Loads NAS-Bench-201 accuracy from `data/nasbench201_full_mapping.pkl`
4. Runs multiple independent seeds for both algorithms
5. Extracts the final Pareto archives
6. Computes **Hypervolume (HV)** and **IGD** against a reference (‘theoretical’) Pareto front
7. Plots both Pareto fronts + the reference front

Notes:
- The niche partitioning is defined for **NAS-Bench-201** only.
- If `data/nasbench201_full_mapping.pkl` is not present, the notebook still runs in hardware-only mode.

In [ ]:
# ===== USER SETTINGS (EDIT THESE) =====

# Git repo URL to clone (must contain HW-NAS-Bench-MetaHeuristicsOptim code).
REPO_URL = "https://github.com/sidab-kh/nsga-II-nas-implementation/tree/nsga-II-part.git"  # TODO: replace
REPO_DIRNAME = "nsga-II-nas-implementation"  # folder inside repo (usually this)

# NAS-Bench-201 accuracy mapping (optional).
# If present, enables the 'error' objective without needing the 4GB .pth benchmark file.
NB201_MAPPING_REL_PATH = "data/nasbench201_full_mapping.pkl"

# Experiment parameters
SEARCH_SPACE = "nasbench201"  # required
DEVICE = "edgegpu"
DATASET = "cifar10"

NUM_RUNS = 10
POPULATION_TOTAL = 60   # total pop budget; partitioned splits across 6 niches
ITERATIONS = 50
SEED_BASE = 42

# Objective pair for metrics/plots (minimization space):
# - ('latency', 'energy') is fully supported from HW pickle only
# - any pair including 'error' requires `data/nasbench201_full_mapping.pkl`
OBJECTIVE_PAIR = ("latency", "energy")

# Reference front computation:
# For ('latency','energy'): FULL is fast (15625 points).
# If you include 'error', full evaluation requires loading accuracy for many architectures and may be slower.
THEORETICAL_EVAL_LIMIT = None  # None = full space; or set e.g. 3000 for faster reference

# =====================================

In [ ]:
# Install runtime dependencies (Kaggle has many already; these are lightweight).
!pip -q install pymoo seaborn tqdm

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', '-q', 'install', 'nas-bench-201'], returncode=0)

In [ ]:
import os
import re
import sys
import json
import math
import time
import pickle
import subprocess
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from pymoo.indicators.hv import HV
from pymoo.indicators.igd import IGD

sns.set_context('talk')
plt.rcParams['figure.figsize'] = (10, 7)

WORKDIR = Path('/kaggle/working')
WORKDIR

PosixPath('/kaggle/working')

In [16]:
!git clone --branch nsga-II-part --single-branch https://github.com/sidab-kh/nsga-II-nas-implementation.git

fatal: destination path 'nsga-II-nas-implementation' already exists and is not an empty directory.


In [17]:
project_root = WORKDIR / REPO_DIRNAME
if not project_root.exists():
    raise FileNotFoundError(f'Expected project folder not found: {project_root}')

# Add project to import path (avoid packaging; pyproject requires Python>=3.13).
sys.path.insert(0, str(project_root))
print('Project root:', project_root)

Project root: /kaggle/working/nsga-II-nas-implementation


In [ ]:
# Fetch benchmark files into repo's data/ folder
data_dir = project_root / 'data'
pickle_path = data_dir / 'HW-NAS-Bench-v1_0.pickle'
nb201_mapping_path = project_root / NB201_MAPPING_REL_PATH

data_dir.mkdir(parents=True, exist_ok=True)

# HW-NAS-Bench hardware metrics (small enough to download directly)
subprocess.run(
    ['wget', '-q', '-O', str(pickle_path), 'https://raw.githubusercontent.com/GATECH-EIC/HW-NAS-Bench/main/HW-NAS-Bench-v1_0.pickle'],
    check=True,
 )

# NAS-Bench-201 accuracy mapping is expected to be present in the repo.
# If it's missing, we continue in hardware-only mode (accuracy/error objective disabled).
if nb201_mapping_path.exists():
    print('NB201 mapping found:', nb201_mapping_path)
else:
    print('NB201 mapping NOT found (hardware-only):', nb201_mapping_path)

print('Files:', pickle_path.exists(), nb201_mapping_path.exists())
print('HW pickle size (MB):', round(pickle_path.stat().st_size/1e6, 1))
if nb201_mapping_path.exists():
    print('NB201 mapping size (MB):', round(nb201_mapping_path.stat().st_size/1e6, 1))

Downloading: NAS-Bench-201-v1_1-096897.pth
NB201 .pth download failed (continuing hardware-only): Failed to retrieve file url:

	Too many users have viewed or downloaded this file recently. Please
	try accessing the file again later. If the file you are trying to
	access is particularly large or is shared with many people, it may
	take up to 24 hours to be able to view or download the file. If you
	still can't access a file after 24 hours, contact your domain
	administrator.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=16Y0UwGisiouVRxW-W5hEtbxmcHw_0hF_

but Gdown can't. Please check connections and permissions.
Files: True False
HW pickle size (MB): 10.8


In [ ]:
from core.api import HWNASBenchAPI
from core.fitness import HardwareAwareFitness, NASBench201PickleAccuracyAPI
from core.types import VALID_DATASETS, VALID_DEVICES
from algorithms.metaheuristics import get_algorithm
from algorithms.multiobjective.pareto import fast_non_dominated_sort

assert SEARCH_SPACE == 'nasbench201', 'This notebook supports partitioned niches only for nasbench201.'
assert DEVICE in VALID_DEVICES, f'Invalid DEVICE. Valid: {VALID_DEVICES}'
assert DATASET in VALID_DATASETS, f'Invalid DATASET. Valid: {VALID_DATASETS}'

api = HWNASBenchAPI(pickle_path, search_space=SEARCH_SPACE)
api

HWNASBenchAPI(search_space=<SearchSpace.NASBENCH201: 'nasbench201'>, size=15625)

In [ ]:
# Fitness object is used to compute normalized objectives consistent with the algorithms.
fitness = HardwareAwareFitness(
    api,
    nb201_mapping_path=str(nb201_mapping_path) if nb201_mapping_path.exists() else None,
    target_device=DEVICE,
    dataset=DATASET,
    latency_weight=0.5,
    energy_weight=0.2,
    accuracy_weight=0.3,
 )
print(fitness)

HardwareAwareFitness(device='edgegpu', dataset='cifar10', w_lat=0.71, w_eng=0.29, w_acc=0.00 [OFF])


/kaggle/working/nsga-II-nas-implementation/core/fitness.py:191: UserWarning: accuracy_weight > 0 but NAS-Bench-201 is unavailable. Forcing accuracy_weight=0 and running in hardware-only mode.
  warnings.warn(


In [ ]:
def select_objective_columns(objectives_3d: np.ndarray, pair: tuple[str, str]) -> np.ndarray:
    # The repo uses compute_multi() -> (lat_norm, eng_norm, err_norm), all minimization
    idx = {'latency': 0, 'energy': 1, 'error': 2}
    a, b = pair
    # Allow using 'accuracy' as an alias: we represent it as 'error' (minimization) = 1 - acc/acc_baseline
    if a == 'accuracy':
        a = 'error'
    if b == 'accuracy':
        b = 'error'
    if a not in idx or b not in idx:
        raise ValueError(f'OBJECTIVE_PAIR must use keys {list(idx)} (plus alias accuracy), got {pair}')
    return objectives_3d[:, [idx[a], idx[b]]]

def nondominated_front(points: np.ndarray) -> np.ndarray:
    # Progress bar here shows the Pareto filtering cost itself, which can dominate runtime.
    points = np.asarray(points, dtype=float)
    if len(points) == 0:
        return points
    if len(points) == 1:
        return points.copy()

    is_dominated = np.zeros(len(points), dtype=bool)
    for i in tqdm(range(len(points)), desc='Nondominated front search', total=len(points)):
        if is_dominated[i]:
            continue
        for j in range(len(points)):
            if i == j or is_dominated[i]:
                continue
            # If point j dominates point i, i cannot be on the first front.
            if np.all(points[j] <= points[i]) and np.any(points[j] < points[i]):
                is_dominated[i] = True
                break

    return points[~is_dominated]

def compute_reference_front_from_hw_pickle(
    pickle_path: Path,
    dataset: str,
    device: str,
    fitness_obj: HardwareAwareFitness,
    eval_limit: int | None = None,
    use_accuracy: bool = False,
    nb201_mapping_path: Path | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns (ref_front_2d, all_points_2d, all_points_3d_norm).

    - all_points_3d_norm is (lat_norm, eng_norm, err_norm) for the evaluated set
    - all_points_2d is the selected objective-pair projection
    - ref_front_2d is the non-dominated set on all_points_2d
    """
    data = pickle.load(open(pickle_path, 'rb'))
    nb = data['nasbench201'][dataset]

    lat_key = f'{device}_latency'
    eng_key = f'{device}_energy'
    if lat_key not in nb:
        raise KeyError(f'Missing {lat_key} in pickle for {dataset}')

    lat = np.asarray(nb[lat_key], dtype=float)
    if eng_key in nb:
        eng = np.asarray(nb[eng_key], dtype=float)
    else:
        # Some devices might not have energy; use zeros (still allows latency-only tradeoffs).
        eng = np.zeros_like(lat)

    n = len(lat)
    idxs = np.arange(n)
    if eval_limit is not None and eval_limit < n:
        rng = np.random.default_rng(0)
        idxs = rng.choice(idxs, size=int(eval_limit), replace=False)
        idxs.sort()

    print(f'Finding theoretical Pareto front on {len(idxs)} architectures...')

    lat_list = []
    eng_list = []
    err_list = []

    acc_api = None
    if use_accuracy and nb201_mapping_path is not None and nb201_mapping_path.exists():
        acc_api = NASBench201PickleAccuracyAPI(str(nb201_mapping_path))
        if not acc_api.available:
            acc_api = None

    if use_accuracy and acc_api is None:
        print('Accuracy unavailable for reference front (using constant error=1).')

    for arch_idx in tqdm(idxs, desc='Reference front search', total=len(idxs)):
        lat_list.append(float(lat[int(arch_idx)]))
        eng_list.append(float(eng[int(arch_idx)]))
        if use_accuracy and acc_api is not None:
            acc = float(acc_api.get_accuracy(int(arch_idx), dataset=dataset))
            err_list.append(1.0 - (acc / fitness_obj.acc_baseline if fitness_obj.acc_baseline > 0 else 1.0))
        else:
            err_list.append(1.0)

    lat_arr = np.asarray(lat_list, dtype=float) / fitness_obj.lat_baseline if fitness_obj.lat_baseline > 0 else np.asarray(lat_list, dtype=float)
    eng_arr = np.asarray(eng_list, dtype=float) / fitness_obj.eng_baseline if fitness_obj.eng_baseline > 0 else np.asarray(eng_list, dtype=float)
    err_arr = np.asarray(err_list, dtype=float)

    all_3d = np.vstack([lat_arr, eng_arr, err_arr]).T
    all_2d = select_objective_columns(all_3d, OBJECTIVE_PAIR)
    ref_2d = nondominated_front(all_2d)
    return ref_2d, all_2d, all_3d

use_acc_for_ref = any(o in ('error', 'accuracy') for o in OBJECTIVE_PAIR)
ref_front_2d, all_points_2d, all_points_3d = compute_reference_front_from_hw_pickle(
    pickle_path=pickle_path,
    dataset=DATASET,
    device=DEVICE,
    fitness_obj=fitness,
    eval_limit=THEORETICAL_EVAL_LIMIT,
    use_accuracy=use_acc_for_ref,
    nb201_mapping_path=nb201_mapping_path,
 )
print('Reference front size:', len(ref_front_2d), 'out of', len(all_points_2d))

Finding theoretical Pareto front on 3000 architectures...


Reference front search:   0%|          | 0/3000 [00:00<?, ?it/s]

Nondominated front search:   0%|          | 0/3000 [00:00<?, ?it/s]

Reference front size: 2 out of 3000


In [23]:
# Hypervolume reference point: must be worse than all points (minimization).
max_vals = np.max(all_points_2d, axis=0)
min_vals = np.min(all_points_2d, axis=0)
ref_point = max_vals + 0.1 * (max_vals - min_vals + 1e-12)
print('ref_point:', ref_point)

hv = HV(ref_point=ref_point)
igd = IGD(ref_front_2d)

ref_point: [0.12274651 0.93572842]


In [ ]:
def run_one(algo_key: str, seed: int) -> dict:
    AlgoClass = get_algorithm(algo_key)

    # New fitness per run so query statistics are per-run (but the API is a fast lookup).
    fit = HardwareAwareFitness(
        api,
        nb201_mapping_path=str(nb201_mapping_path) if nb201_mapping_path.exists() else None,
        target_device=DEVICE,
        dataset=DATASET,
        latency_weight=0.5,
        energy_weight=0.2,
        accuracy_weight=0.3,
    )

    optimizer = AlgoClass(
        fitness_function=fit,
        search_space_size=api.search_space_size(DATASET),
        dim=api.search_dimension(DATASET),
        population_size=POPULATION_TOTAL,
        max_iterations=ITERATIONS,
        seed=seed,
    )

    t0 = time.perf_counter()
    _best_sol, best_fit, _hist = optimizer.optimize()
    elapsed = time.perf_counter() - t0

    # Extract Pareto archive from optimizer if available
    arch_obj = getattr(optimizer, 'archive_objectives', None)
    if arch_obj is None or len(arch_obj) == 0:
        raise RuntimeError(f'{algo_key} did not expose archive_objectives')

    front3 = np.asarray(arch_obj, dtype=float)
    front2 = select_objective_columns(front3, OBJECTIVE_PAIR)
    front2 = front2[np.all(np.isfinite(front2), axis=1)]
    front2 = nondominated_front(front2)

    if len(front2) == 0:
        hv_val = 0.0
        igd_val = float('inf')
    else:
        hv_val = float(hv(front2))
        igd_val = float(igd(front2))

    return {
        'algo': algo_key,
        'seed': seed,
        'elapsed_s': elapsed,
        'best_fitness': float(best_fit),
        'front2': front2,
        'hv': hv_val,
        'igd': igd_val,
        'queries': int(fit.get_statistics()['api_queries']),
        'archive_size': int(len(front3)),
    }

ALGOS = ['NSGA2', 'NSGA2-PARTITIONED']
results = []
for algo in ALGOS:
    for r in range(NUM_RUNS):
        seed = SEED_BASE + r
        print(f'Running {algo} seed={seed} ...')
        res = run_one(algo, seed)
        results.append(res)
        print(
            f"  hv={res['hv']:.6g} igd={res['igd']:.6g} "
            f"archive={res['archive_size']} elapsed={res['elapsed_s']:.1f}s queries={res['queries']}"
        )

Running NSGA2 seed=42 ...


KeyboardInterrupt: 

In [ ]:
# Summaries
rows = []
for r in results:
    rows.append({k: v for k, v in r.items() if k not in {'front2'}})
df = pd.DataFrame(rows)
df

In [ ]:
summary = df.groupby('algo').agg(
    hv_mean=('hv','mean'),
    hv_std=('hv','std'),
    igd_mean=('igd','mean'),
    igd_std=('igd','std'),
    elapsed_mean=('elapsed_s','mean'),
    queries_mean=('queries','mean'),
).reset_index()
summary

In [ ]:
# Boxplots for HV and IGD across runs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='algo', y='hv', ax=axes[0])
axes[0].set_title('Hypervolume (higher is better)')
axes[0].set_xlabel('')

sns.boxplot(data=df, x='algo', y='igd', ax=axes[1])
axes[1].set_title('IGD (lower is better)')
axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Choose the best run per algorithm (by HV) for plotting
best_by_algo = {}
for algo in ALGOS:
    sub = [r for r in results if r['algo'] == algo]
    best = max(sub, key=lambda x: x['hv'])
    best_by_algo[algo] = best
best_by_algo

In [ ]:
# Pareto front plot (2D)
x_name, y_name = OBJECTIVE_PAIR

def pretty_axis(name: str) -> str:
    if name == 'latency':
        return 'latency (normalized, lower better)'
    if name == 'energy':
        return 'energy (normalized, lower better)'
    if name == 'error':
        return 'error = 1 - acc/acc_baseline (lower better)'
    if name == 'accuracy':
        return 'accuracy (represented as error, lower better)'
    return f'{name} (lower better)'

def aggregate_front_for_algo(algo: str) -> np.ndarray:
    pts = [r['front2'] for r in results if r['algo'] == algo]
    if not pts:
        return np.empty((0, 2))
    union = np.vstack(pts)
    return nondominated_front(union)

plt.figure(figsize=(10, 7))
plt.scatter(ref_front_2d[:, 0], ref_front_2d[:, 1], s=18, c='gray', alpha=0.5, label='Reference (theoretical) Pareto')

colors = {'NSGA2': 'tab:red', 'NSGA2-PARTITIONED': 'tab:blue'}
for algo in ALGOS:
    # Best single run (by HV)
    best_front = best_by_algo[algo]['front2']
    plt.scatter(best_front[:, 0], best_front[:, 1], s=30, c=colors[algo], alpha=0.9, label=f'{algo} (best HV)')
    # Aggregated (union across runs, then non-dominated)
    agg_front = aggregate_front_for_algo(algo)
    if len(agg_front) > 0:
        plt.scatter(agg_front[:, 0], agg_front[:, 1], s=30, facecolors='none', edgecolors=colors[algo], linewidths=1.5, alpha=0.9, label=f'{algo} (union ND)')

plt.title(f'Pareto fronts in normalized objective space: {OBJECTIVE_PAIR}')
plt.xlabel(pretty_axis(x_name))
plt.ylabel(pretty_axis(y_name))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Save outputs
out_dir = WORKDIR / 'nsga2_partitioned_comparison'
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / 'per_run_metrics.csv', index=False)
summary.to_csv(out_dir / 'summary_metrics.csv', index=False)

with open(out_dir / 'settings.json', 'w') as f:
    json.dump({
        'REPO_URL': REPO_URL,
        'SEARCH_SPACE': SEARCH_SPACE,
        'DEVICE': DEVICE,
        'DATASET': DATASET,
        'NUM_RUNS': NUM_RUNS,
        'POPULATION_TOTAL': POPULATION_TOTAL,
        'ITERATIONS': ITERATIONS,
        'SEED_BASE': SEED_BASE,
        'OBJECTIVE_PAIR': OBJECTIVE_PAIR,
        'THEORETICAL_EVAL_LIMIT': THEORETICAL_EVAL_LIMIT,
        'ref_point': ref_point.tolist(),
        'reference_front_size': int(len(ref_front_2d)),
    }, f, indent=2)

print('Saved to:', out_dir)
list(out_dir.iterdir())